In [1]:
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('nyc-taxi-experiment')

2025/12/29 16:17:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 16:17:55 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 16:17:55 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 16:17:55 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 16:17:55 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 16:17:55 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/experiment_tracking/mlruns/1', creation_time=1766823082109, experiment_id='1', last_update_time=1766823082109, lifecycle_stage='active', name='nyc-taxi-experiment', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [2]:
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error




/home/codespace/anaconda3/envs/exp_tracking/lib/python3.14/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [11]:
df_train = read_dataframe('./data/green_tripdata_2025-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2025-02.parquet')

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [12]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [ ]:

with mlflow.start_run():
    n_estimators = 100
    gbr = GradientBoostingRegressor(random_state=10, n_estimators=n_estimators)
    mlflow.set_tag('developer', 'Victor')
    mlflow.log_param('n_estimators', n_estimators)
    mlflow.log_param('training_data', './data/green_tripdata_2025-01.parquet')
    mlflow.log_param('testing_data', './data/green_tripdata_2025-02.parquet')
    gbr.fit(X_train, y_train)
    y_pred = gbr.predict(X_val)
    mse = mean_squared_error(y_pred, y_val)
    mlflow.log_metric('rmse', mse)

In [ ]:

def objective(params):
    with mlflow.start_run(): 
        mlflow.set_tag('model', 'GradientBoost')
        mlflow.log_params(params)
        gbr = GradientBoostingRegressor(**params, n_iter_no_change=10, validation_fraction=0.1)
        gbr.fit(X_train, y_train)
        y_pred = gbr.predict(X_val)
        rmse = mean_squared_error(y_pred, y_val)
        mlflow.log_metric('rmse', rmse)
        mlflow.sklearn.log_model(gbr, "model")

    return {'status': STATUS_OK, 'loss':rmse}

In [12]:
search_space = {
    'n_estimators': scope.int(hp.quniform('n_estimators', 100, 500, 10)),
    'learning_rate': hp.loguniform('learning_rate', -3, -1),
    'max_depth': scope.int(hp.quniform('max_depth', 3, 10, 1)),
    'subsample': hp.quniform('subsample', 0.5, 1, 0.1),
    'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1))
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

2025/12/29 07:24:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 10%|█         | 1/10 [00:08<01:16,  8.45s/trial, best loss: 30.83446232148319]

2025/12/29 07:24:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 20%|██        | 2/10 [00:20<01:26, 10.82s/trial, best loss: 29.742608464107214]

...

 80%|████████  | 8/10 [02:13<00:40, 20.46s/trial, best loss: 29.742608464107214]

2025/12/29 07:26:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 90%|█████████ | 9/10 [02:35<00:20, 20.73s/trial, best loss: 29.742608464107214]

2025/12/29 07:26:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 10/10 [02:42<00:00, 16.21s/trial, best loss: 29.742608464107214]


In [13]:
best_result

{'learning_rate': np.float64(0.2180465864731468),
 'max_depth': np.float64(8.0),
 'min_samples_split': np.float64(7.0),
 'n_estimators': np.float64(140.0),
 'subsample': np.float64(0.6000000000000001)}

In [14]:
mlflow.sklearn.autolog(exclusive=False, log_models=False)

In [17]:
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error
def objective_svm(params):
    with mlflow.start_run():
        svr = SVR(**params)
        svr.fit(X_train, y_train)
        y_pred = svr.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_metric('rmse', rmse)
    return {'status': STATUS_OK, 'loss': rmse}

In [18]:
search_space_svm = {
    'C': hp.loguniform('C',-2, 3),
    'epsilon': hp.loguniform('epsilon', -3, 0),
    'gamma': hp.loguniform('gamma', -4, 1)
}

best_result_svm = fmin(
    fn=objective_svm,
    space=search_space_svm,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

100%|██████████| 10/10 [25:50<00:00, 155.07s/trial, best loss: 5.613954268809058]


In [ ]:

with mlflow.start_run():
    svr = SVR(**best_result_svm)
    svr.fit(X_train, y_train)
    
    y_pred = svr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(svr, artifact_path='models/')
    
    print(f"Final Model Saved with RMSE: {rmse}")

Final Model Saved with RMSE: 5.613954268809058


In [ ]:
mlflow.sklearn.autolog(exclusive=False, log_models=False)
def objective_gbr(params):
    with mlflow.start_run(): 
        gbr = GradientBoostingRegressor(**params, n_iter_no_change=10, validation_fraction=0.1)
        gbr.fit(X_train, y_train)
        y_pred = gbr.predict(X_val)
        rmse = root_mean_squared_error(y_pred, y_val)
        mlflow.log_metric('rmse', rmse)
        

    return {'status': STATUS_OK, 'loss':rmse}

search_space_gbr = {
    'n_estimators': scope.int(hp.quniform('n_estimators', 100, 500, 10)),
    'learning_rate': hp.loguniform('learning_rate', -3, -1),
    'max_depth': scope.int(hp.quniform('max_depth', 3, 10, 1)),
    'subsample': hp.quniform('subsample', 0.5, 1, 0.1),
    'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1))
}

best_result_gbr= fmin(
    fn=objective_gbr,
    space=search_space_gbr,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

final_params = best_result_gbr.copy()
final_params['n_estimators'] = int(final_params['n_estimators'])
final_params['max_depth'] = int(final_params['max_depth'])
final_params['min_samples_split'] = int(final_params['min_samples_split'])

with mlflow.start_run():
    gbr = GradientBoostingRegressor(**final_params, n_iter_no_change=10)
    gbr.fit(X_train, y_train)
    
    y_pred = gbr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(gbr, name='models')
    
    print(f"Final Model Saved with RMSE: {rmse}")

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

100%|██████████| 10/10 [03:41<00:00, 22.15s/trial, best loss: 5.443119994840233]


2025/12/29 10:46:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MlflowException: Invalid model name ('models/') provided. Model name must be a non-empty string and cannot contain the following characters: ('/', ':', '.', '%', '"', "'")

In [15]:
final_params = best_result_gbr.copy()
final_params['n_estimators'] = int(final_params['n_estimators'])
final_params['max_depth'] = int(final_params['max_depth'])
final_params['min_samples_split'] = int(final_params['min_samples_split'])

with mlflow.start_run():
    gbr = GradientBoostingRegressor(**final_params, n_iter_no_change=10)
    gbr.fit(X_train, y_train)
    
    y_pred = gbr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(gbr, name='models')
    
    print(f"Final Model Saved with RMSE: {rmse}")

Final Model Saved with RMSE: 5.439791072743411


In [ ]:
import xgboost
train = xgboost.DMatrix(X_train, label=y_train)
valid = xgboost.DMatrix(X_val, label=y_val)

mlflow.xgboost.autolog(exclusive=False, log_models=False)
def objective_xgb(params):
    with mlflow.start_run(nested=True):
        mlflow.set_tag('model', 'xgboost')
        mlflow.log_params(params)
        booster = xgboost.train(params=params, dtrain=train, num_boost_round=500, evals=[(valid, 'validation')])
        y_pred = booster.predict(valid)
        rmse = mean_squared_error(y_val, y_pred)
        mlflow.log_metric('rmse', rmse)
    
    return {'loss': rmse, 'status':STATUS_OK }

search_space_xgb = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result_xgb = fmin(
    fn=objective_xgb,
    space=search_space_xgb,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

final_params = best_result_xgb.copy()
final_params['max_depth'] = int(final_params['max_depth'])
final_params['seed'] = 42
with mlflow.start_run():
    xgb = xgboost.train(
        params=final_params, 
        dtrain=train, 
        num_boost_round=500, 
        evals=[(valid, 'validation')],
        early_stopping_rounds=10
    )
    
    y_pred = xgb.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.xgboost.log_model(xgb, name='models')
    
    print(f"Final Model Saved with RMSE: {rmse}")

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

/home/codespace/anaconda3/envs/exp_tracking/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [11:23:08] WARNING: /workspace/src/objective/regression_obj.cu:282: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)



[0]	validation-rmse:8.41808                           
[1]	validation-rmse:7.84555                           
[2]	validation-rmse:7.37137                           
[3]	validation-rmse:6.98148                           
[4]	validation-rmse:6.66390                           
[5]	validation-rmse:6.40702                           
[6]	validation-rmse:6.19971                           
[7]	validation-rmse:6.03331                           
[8]	validation-rmse:5.90162                           
[9]	validation-rmse:5.79705                           
[10]	validation-rmse:5.71330                          
[11]	validation-rmse:5.64707                          
[12]	validation-rmse:5.59518                          
[13]	validation-rmse:5.55415                          
[14]	validation-rmse:5.52259                          
[15]	validation-rmse:5.49591                          
[16]	validation-rmse:5.47463                          
[17]	validation-rmse:5.45814                          
[18]	valid

/home/codespace/anaconda3/envs/exp_tracking/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [11:24:47] WARNING: /workspace/src/objective/regression_obj.cu:282: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)



[0]	validation-rmse:8.46243                                                    
[1]	validation-rmse:7.91857                                                    
[2]	validation-rmse:7.46309                                                    
[3]	validation-rmse:7.08348                                                    
[4]	validation-rmse:6.76871                                                    
[5]	validation-rmse:6.51000                                                    
[6]	validation-rmse:6.29721                                                    
[7]	validation-rmse:6.12313                                                    
[8]	validation-rmse:5.98158                                                    
[9]	validation-rmse:5.86737                                                    
[10]	validation-rmse:5.77545                                                   
[11]	validation-rmse:5.70080                                                   
[12]	validation-rmse:5.64133            

...

/home/codespace/anaconda3/envs/exp_tracking/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [11:35:10] WARNING: /workspace/src/objective/regression_obj.cu:282: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)



[0]	validation-rmse:8.63740                                                     
[1]	validation-rmse:8.22165                                                     
[2]	validation-rmse:7.85430                                                     
[3]	validation-rmse:7.53091                                                     
[4]	validation-rmse:7.24716                                                     
[5]	validation-rmse:6.99858                                                     
[6]	validation-rmse:6.78129                                                     
[7]	validation-rmse:6.59238                                                     
[8]	validation-rmse:6.42871                                                     
[9]	validation-rmse:6.28640                                                     
[10]	validation-rmse:6.16384                                                    
[11]	validation-rmse:6.05844                                                    
[12]	validation-rmse:5.96714

/home/codespace/anaconda3/envs/exp_tracking/lib/python3.14/site-packages/xgboost/training.py:199: UserWarning: [11:36:20] WARNING: /workspace/src/objective/regression_obj.cu:282: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)



[0]	validation-rmse:6.62928                                                     
[1]	validation-rmse:5.81739                                                     
[2]	validation-rmse:5.62492                                                     
[3]	validation-rmse:5.55379                                                     
[4]	validation-rmse:5.54440                                                     
[5]	validation-rmse:5.54021                                                     
[6]	validation-rmse:5.54006                                                     
[7]	validation-rmse:5.53748                                                     
[8]	validation-rmse:5.52565                                                     
[9]	validation-rmse:5.52540                                                     
[10]	validation-rmse:5.52707                                                    
[11]	validation-rmse:5.52472                                                    
[12]	validation-rmse:5.52222

TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'scipy.sparse._csr.csr_matrix'>)

In [2]:
final_params = best_result_xgb.copy()
final_params['max_depth'] = int(final_params['max_depth'])
final_params['seed'] = 42
with mlflow.start_run():
    xgb = xgboost.train(
        params=final_params, 
        dtrain=train, 
        num_boost_round=500, 
        evals=[(valid, 'validation')],
        early_stopping_rounds=10
    )
    
    y_pred = xgb.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.xgboost.log_model(xgb, name='models')
    
    print(f"Final Model Saved with RMSE: {rmse}")


NameError: name 'best_result_xgb' is not defined

In [3]:
from mlflow.tracking import MlflowClient
MLFLOW_TRACKING_URI = 'sqlite:///mlflow.db'
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [4]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/experiment_tracking/mlruns/2', creation_time=1767009432605, experiment_id='2', last_update_time=1767009432605, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/experiment_tracking/mlruns/1', creation_time=1766823082109, experiment_id='1', last_update_time=1766823082109, lifecycle_stage='active', name='nyc-taxi-experiment', tags={'mlflow.experimentKind': 'custom_model_development'}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1766816793039, experiment_id='0', last_update_time=1766816793039, lifecycle_stage='active', name='Default', tags={}>]

In [11]:
client.create_experiment('my-cool-experiment')

'2'

In [5]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',filter_string='', run_view_type=ViewType.ACTIVE_ONLY, max_results=5, order_by= ['metrics.rmse ASC']
)

In [6]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f} ")

run id: 7cd55f7b6caa4ca7944a5a14a31a25e7, rmse: 5.4318 
run id: 72ca808dfc6c4fbd8c76933c450021b9, rmse: 5.4398 
run id: 7f0b86bd641e45a6833ae3130d508a5d, rmse: 5.4431 
run id: 64c84517cdfe4a21aa85af6e6e87b49c, rmse: 5.4682 
run id: e2873a1f2f314c0d9ad413ce4d821e03, rmse: 5.4696 


In [7]:
run_id = '7cd55f7b6caa4ca7944a5a14a31a25e7'
model_uri = f"runs:/{run_id}/model"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.register_model(model_uri=model_uri, name='nyc-taxi-regressor')

2025/12/29 16:20:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 16:20:10 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 16:20:10 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 16:20:10 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Successfully registered model 'nyc-taxi-regressor'.
2025/12/29 16:20:11 WARNING mlflow.tracking._model_registry.fluent: Run with id 7cd55f7b6caa4ca7944a5a14a31a25e7 has no artifacts at artifact path 'model', registering model based on models:/m-8c1f4e2064544f569c4d89bed436f98b instead
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1767025211046, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1767025211046, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='7cd55f7b6caa4ca7944a5a14a31a25e7', run_link=None, source='models:/m-8c1f4e2064544f569c4d89bed436f98b', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [9]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1767025211003, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1767025211046, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1767025211046, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1767025211046, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='7cd55f7b6caa4ca7944a5a14a31a25e7', run_link=None, source='models:/m-8c1f4e2064544f569c4d89bed436f98b', status='READY', status_message=None, tags={}, user_id=None, version=1>], name='nyc-taxi-regressor', tags={}>,
 <RegisteredModel: aliases={}, creation_timestamp=1767008053313, deployment_job_id=None, deployment_job_state=None, description='', last_updated_timestamp=1767008837942, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1767008071796, current_stage='Staging', deployment_job_state=None, description='', last_updated_timestamp=1767008551382, m

In [12]:
model_name = 'nyc-trip-regressor'
latest_versions = client.get_latest_versions(name=model_name)
for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 3, stage: Staging
version: 2, stage: None


/tmp/ipykernel_10391/3860113192.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [13]:
client.transition_model_version_stage(name=model_name, version=2, stage='Production', archive_existing_versions=False)

/tmp/ipykernel_10391/2283156632.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name=model_name, version=2, stage='Production', archive_existing_versions=False)


<ModelVersion: aliases=[], creation_timestamp=1767008063170, current_stage='Production', deployment_job_state=None, description='', last_updated_timestamp=1767025971014, metrics=None, model_id=None, name='nyc-trip-regressor', params=None, run_id='72ca808dfc6c4fbd8c76933c450021b9', run_link='', source='models:/m-0451c5a6d2ae434cb9e5c9327b94a3e2', status='READY', status_message=None, tags={}, user_id=None, version=2>